#configuração

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline


#carregando dataset


In [ ]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip'
import zipfile, io, requests

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
# Escolher o arquivo com todos os exemplos:
df = pd.read_csv(z.open('bank-additional/bank-additional-full.csv'), sep=';')

#vizualizaçao inicial

In [ ]:
print("Formato:", df.shape)
print("Colunas:", list(df.columns))
print(df.head())

#análise exploratória básica

In [ ]:
print(df['y'].value_counts())
print(df.describe())

plt.bar(['Não','Sim'], df['y'].map({'no':0, 'yes':1}).value_counts().loc[[0,1]].values,
        color=['lightcoral','lightgreen'])
plt.title('Distribuição da resposta (y)')
plt.xlabel('Resposta')
plt.ylabel('Quantidade')
plt.show()

#pré-processamento

In [ ]:

df_processed = df.copy()

# codificando target
df_processed['y'] = df_processed['y'].map({'yes':1, 'no':0})

# identificando colunas categóricas
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if 'y' in categorical_cols:
    categorical_cols.remove('y')  # só remove se existir

# aplicando LabelEncoder nas categóricas
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le


#separação separação e dividisao de variaveis,  treino/teste + normalização

In [ ]:

# separand features/target
X = df_processed.drop('y', axis=1)
y = df_processed['y']

# divisao
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# normalizando colunas numéricas
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
# removend colunas que são numéricas mas derivadas de codificação categórica
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])


#treinamento dos modelos

In [ ]:

models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'NaiveBayes': GaussianNB(),
    'Árvore': DecisionTreeClassifier(max_depth=4, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, zero_division=0)
    results[name] = {'model': model, 'accuracy': acc, 'report': report}
    print(f"{name} — Acurácia: {acc:.4f}")


#Árvore de Decisão

In [ ]:

dt = results['Árvore']['model']
feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)
print(feat_imp)

plt.barh(feat_imp['feature'], feat_imp['importance'], color='skyblue')
plt.title('Importância das características — Árvore')
plt.tight_layout()
plt.show()


#Naive Bayes

In [ ]:

nb = results['NaiveBayes']['model']
print("Médias por classe (θ) — Classe 0")
for i, feat in enumerate(X.columns):
    print(f"{feat:25s}: {nb.theta_[0][i]:.2f}")
print("\nClasse 1:")
for i, feat in enumerate(X.columns):
    print(f"{feat:25s}: {nb.theta_[1][i]:.2f}")


#KNN

In [ ]:

knn = results['KNN']['model']
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1]
distances, indices = knn.kneighbors(sample)
print("Classe real:", y_test.iloc[sample_idx])
print("Prevista:", knn.predict(sample)[0])
print("Vizinhos mais próximos:")
for i, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    print(f"{i+1}. Distância: {dist:.2f}, Classe: {y_train.iloc[idx]}")


#resutados finais

In [ ]:

for name, info in results.items():
    print(f"{name}: acurácia = {info['accuracy']:.4f}")
    print(info['report'])
